In [ ]:
#============================================================
#ANALISIS Y SEPARACION DE EEG MEDIANTE FOURIER
#BITalino + OpenSignals
#============================================================
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
#============================================================
#1. UBICACION DE CARPETAS
#============================================================
#Carpeta donde se encuentra este archivo .py
CARPETA_CODIGO = Path(__file__).resolve().parent
#Subimos a Lab_06
CARPETA_LA =CARPETA_CODIGO.parent
#Carpeta donde estan los archivos EEG
CARPETA_DATOS=CARPETA_LAB / "Datos_EEG"
#Carpeta donde se guardaran los resultados
CARPETA_RESULTADOS =CARPETA_LAB / "Resultados_EEG"
# Crear carpeta de resultados si no existe
CARPETA_RESULTADOS.mkdir(exist_ok=True)
# ============================================================
# 2. CONFIGURACION
# ============================================================
# Frecuencia de muestreo de BITalino
FS = 1000  # Hz

# Tiempo que se mostrara en las graficas temporales
SEGUNDOS_GRAFICA = 30

# ============================================================
# 3. BANDAS EEG
# ============================================================

# Se utiliza 0.8 Hz como limite inferior porque
# la guia del sensor EEG indica un pasa-banda de 0.8-48 Hz.

BANDAS = {
    "Delta": (0.8, 4),
    "Theta": (4, 8),
    "Alpha": (8, 12),
    "Beta": (12, 25),
    "Gamma": (25, 48)
}


# ============================================================
# 4. FUNCION PARA CARGAR EL ARCHIVO DE OPENSIGNALS
# ============================================================

def cargar_eeg(ruta_archivo):

    """
    Carga un archivo TXT de OpenSignals.

    Se obtienen las dos ultimas columnas numericas
    como canales EEG A1 y A2.
    """

    filas = []

    try:

        with open(
            ruta_archivo,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as archivo:

            for linea in archivo:

                # Ignorar encabezados
                if linea.startswith("#"):
                    continue

                linea = linea.strip()

                if not linea:
                    continue

                columnas = linea.split("\t")

                valores = []

                for campo in columnas:

                    campo = campo.strip()

                    if campo == "":
                        valores.append(np.nan)

                    else:

                        try:
                            valores.append(float(campo))

                        except ValueError:
                            valores.append(np.nan)

                filas.append(valores)

    except Exception as error:

        print("\nERROR AL LEER:")
        print(ruta_archivo)
        print(error)

        return None, None


    if len(filas) == 0:

        print(
            f"\nERROR: {ruta_archivo.name} "
            "no contiene datos."
        )

        return None, None


    # Numero maximo de columnas
    numero_columnas = max(
        len(fila)
        for fila in filas
    )


    # Crear matriz
    matriz = np.full(
        (len(filas), numero_columnas),
        np.nan
    )


    # Copiar datos
    for i, fila in enumerate(filas):

        matriz[
            i,
            :len(fila)
        ] = fila


    # Buscar columnas que tengan datos numericos
    columnas_validas = []

    for j in range(matriz.shape[1]):

        cantidad_validos = np.sum(
            np.isfinite(matriz[:, j])
        )

        if cantidad_validos > 0:

            columnas_validas.append(j)


    if len(columnas_validas) < 2:

        print(
            f"\nERROR: {ruta_archivo.name} "
            "no tiene dos canales numericos."
        )

        return None, None


    # Las dos ultimas columnas numericas
    indice_A1 = columnas_validas[-2]
    indice_A2 = columnas_validas[-1]


    A1 = matriz[:, indice_A1]
    A2 = matriz[:, indice_A2]


    # Conservar solamente filas completas
    validos = (
        np.isfinite(A1) &
        np.isfinite(A2)
    )


    A1 = A1[validos]
    A2 = A2[validos]


    if len(A1) == 0:

        print(
            f"\nERROR: {ruta_archivo.name} "
            "no contiene muestras validas."
        )

        return None, None


    return A1, A2


# ============================================================
# 5. FUNCION FFT + SEPARACION + IFFT
# ============================================================

def separar_fourier(señal, fs):

    """
    Realiza:

    1. Centrado de la señal
    2. FFT
    3. Seleccion de frecuencias
    4. IFFT para reconstruir cada banda
    """

    # --------------------------------------------------------
    # Eliminar componente DC
    # --------------------------------------------------------

    media = np.mean(señal)

    señal_centrada = señal - media


    # --------------------------------------------------------
    # FFT
    # --------------------------------------------------------

    X = np.fft.rfft(señal_centrada)


    # --------------------------------------------------------
    # Vector de frecuencias
    # --------------------------------------------------------

    frecuencias = np.fft.rfftfreq(
        len(señal_centrada),
        d=1 / fs
    )


    # Diccionarios para guardar resultados
    señales_bandas = {}
    espectros_bandas = {}


    # --------------------------------------------------------
    # Separar cada banda
    # --------------------------------------------------------

    for nombre, (fmin, fmax) in BANDAS.items():

        # Mascara de frecuencias
        mascara = (
            (frecuencias >= fmin) &
            (frecuencias < fmax)
        )


        # Crear espectro vacio
        X_banda = np.zeros_like(X)


        # Conservar solamente la banda
        X_banda[mascara] = X[mascara]


        # Guardar espectro
        espectros_bandas[nombre] = X_banda


        # ----------------------------------------------------
        # IFFT
        # ----------------------------------------------------

        señal_banda = np.fft.irfft(
            X_banda,
            n=len(señal_centrada)
        )


        # Guardar señal reconstruida
        señales_bandas[nombre] = señal_banda


    return (
        frecuencias,
        X,
        señales_bandas,
        espectros_bandas,
        media
    )


# ============================================================
# 6. FUNCION PARA PROCESAR UN ARCHIVO
# ============================================================

def procesar_archivo(ruta_archivo):

    print("\n" + "=" * 70)
    print(f"PROCESANDO: {ruta_archivo.name}")
    print("=" * 70)


    # --------------------------------------------------------
    # Cargar EEG
    # --------------------------------------------------------

    A1, A2 = cargar_eeg(ruta_archivo)


    if A1 is None:

        return


    N = len(A1)

    duracion = N / FS


    print(f"Muestras: {N}")
    print(f"Frecuencia de muestreo: {FS} Hz")
    print(f"Duracion: {duracion:.2f} segundos")


    # --------------------------------------------------------
    # Crear carpeta individual
    # --------------------------------------------------------

    nombre_resultado = ruta_archivo.stem

    carpeta_salida = (
        CARPETA_RESULTADOS /
        nombre_resultado
    )

    carpeta_salida.mkdir(
        parents=True,
        exist_ok=True
    )


    # ========================================================
    # PROCESAR A1
    # ========================================================

    (
        frecuencias_A1,
        X_A1,
        bandas_A1,
        espectros_A1,
        media_A1
    ) = separar_fourier(A1, FS)


    # ========================================================
    # PROCESAR A2
    # ========================================================

    (
        frecuencias_A2,
        X_A2,
        bandas_A2,
        espectros_A2,
        media_A2
    ) = separar_fourier(A2, FS)


    # ========================================================
    # VECTOR DE TIEMPO
    # ========================================================

    tiempo = np.arange(N) / FS


    # ========================================================
    # 1. EEG CRUDA
    # ========================================================

    muestras_grafica = min(
        int(SEGUNDOS_GRAFICA * FS),
        N
    )

    t_grafica = tiempo[:muestras_grafica]


    plt.figure(figsize=(14, 8))


    plt.subplot(2, 1, 1)

    plt.plot(
        t_grafica,
        A1[:muestras_grafica]
    )

    plt.title(
        f"EEG cruda - A1\n{ruta_archivo.stem}"
    )

    plt.xlabel("Tiempo (s)")
    plt.ylabel("Amplitud")
    plt.grid(True)


    plt.subplot(2, 1, 2)

    plt.plot(
        t_grafica,
        A2[:muestras_grafica]
    )

    plt.title("EEG cruda - A2")

    plt.xlabel("Tiempo (s)")
    plt.ylabel("Amplitud")
    plt.grid(True)


    plt.tight_layout()


    plt.savefig(
        carpeta_salida / "01_EEG_cruda.png",
        dpi=300
    )

    plt.close()


    # ========================================================
    # 2. ESPECTRO DE FOURIER
    # ========================================================

    magnitud_A1 = np.abs(X_A1) / N
    magnitud_A2 = np.abs(X_A2) / N


    plt.figure(figsize=(14, 8))


    plt.subplot(2, 1, 1)

    plt.plot(
        frecuencias_A1,
        magnitud_A1
    )

    plt.xlim(0, 50)

    plt.title(
        f"Espectro de Fourier - A1\n"
        f"{ruta_archivo.stem}"
    )

    plt.xlabel("Frecuencia (Hz)")
    plt.ylabel("|X(f)|")
    plt.grid(True)


    plt.subplot(2, 1, 2)

    plt.plot(
        frecuencias_A2,
        magnitud_A2
    )

    plt.xlim(0, 50)

    plt.title("Espectro de Fourier - A2")

    plt.xlabel("Frecuencia (Hz)")
    plt.ylabel("|X(f)|")
    plt.grid(True)


    plt.tight_layout()


    plt.savefig(
        carpeta_salida / "02_espectro_Fourier.png",
        dpi=300
    )

    plt.close()


    # ========================================================
    # 3. BANDAS A1
    # ========================================================

    fig, axes = plt.subplots(
        5,
        1,
        figsize=(14, 12),
        sharex=True
    )


    for i, nombre in enumerate(BANDAS):

        axes[i].plot(
            t_grafica,
            bandas_A1[nombre][:muestras_grafica]
        )

        fmin, fmax = BANDAS[nombre]

        axes[i].set_title(
            f"{nombre} ({fmin}-{fmax} Hz) - A1"
        )

        axes[i].set_ylabel("Amplitud")
        axes[i].grid(True)


    axes[-1].set_xlabel("Tiempo (s)")


    plt.tight_layout()


    plt.savefig(
        carpeta_salida / "03_bandas_A1.png",
        dpi=300
    )

    plt.close()


    # ========================================================
    # 4. BANDAS A2
    # ========================================================

    fig, axes = plt.subplots(
        5,
        1,
        figsize=(14, 12),
        sharex=True
    )


    for i, nombre in enumerate(BANDAS):

        axes[i].plot(
            t_grafica,
            bandas_A2[nombre][:muestras_grafica]
        )

        fmin, fmax = BANDAS[nombre]

        axes[i].set_title(
            f"{nombre} ({fmin}-{fmax} Hz) - A2"
        )

        axes[i].set_ylabel("Amplitud")
        axes[i].grid(True)


    axes[-1].set_xlabel("Tiempo (s)")


    plt.tight_layout()


    plt.savefig(
        carpeta_salida / "04_bandas_A2.png",
        dpi=300
    )

    plt.close()


    # ========================================================
    # 5. COMPARACION DE TODAS LAS BANDAS A1
    # ========================================================

    fig, axes = plt.subplots(
        6,
        1,
        figsize=(14, 14),
        sharex=True
    )


    # EEG original
    axes[0].plot(
        t_grafica,
        A1[:muestras_grafica]
    )

    axes[0].set_title("EEG cruda - A1")
    axes[0].set_ylabel("Amplitud")
    axes[0].grid(True)


    # Bandas
    for i, nombre in enumerate(BANDAS):

        axes[i + 1].plot(
            t_grafica,
            bandas_A1[nombre][:muestras_grafica]
        )

        axes[i + 1].set_title(nombre)
        axes[i + 1].set_ylabel("Amplitud")
        axes[i + 1].grid(True)


    axes[-1].set_xlabel("Tiempo (s)")


    plt.tight_layout()


    plt.savefig(
        carpeta_salida /
        "05_separacion_bandas_A1.png",
        dpi=300
    )

    plt.close()


    # ========================================================
    # 6. POTENCIA DE CADA BANDA
    # ========================================================

    potencia_A1 = {}
    potencia_A2 = {}


    for nombre in BANDAS:

        potencia_A1[nombre] = np.mean(
            bandas_A1[nombre] ** 2
        )

        potencia_A2[nombre] = np.mean(
            bandas_A2[nombre] ** 2
        )


    # --------------------------------------------------------
    # Grafico
    # --------------------------------------------------------

    nombres = list(BANDAS.keys())

    valores_A1 = [
        potencia_A1[nombre]
        for nombre in nombres
    ]

    valores_A2 = [
        potencia_A2[nombre]
        for nombre in nombres
    ]


    x = np.arange(len(nombres))

    ancho = 0.35


    plt.figure(figsize=(10, 6))


    plt.bar(
        x - ancho / 2,
        valores_A1,
        ancho,
        label="A1"
    )


    plt.bar(
        x + ancho / 2,
        valores_A2,
        ancho,
        label="A2"
    )


    plt.xticks(
        x,
        nombres
    )

    plt.xlabel("Banda EEG")
    plt.ylabel("Potencia")
    plt.title(
        f"Potencia de bandas EEG\n"
        f"{ruta_archivo.stem}"
    )

    plt.legend()
    plt.grid(axis="y")


    plt.tight_layout()


    plt.savefig(
        carpeta_salida /
        "06_potencia_bandas.png",
        dpi=300
    )

    plt.close()


    # ========================================================
    # 7. RECONSTRUCCION DE LA EEG MEDIANTE IFFT
    # ========================================================

    EEG_reconstruida_A1 = media_A1

    EEG_reconstruida_A2 = media_A2


    for nombre in BANDAS:

        EEG_reconstruida_A1 = (
            EEG_reconstruida_A1 +
            bandas_A1[nombre]
        )

        EEG_reconstruida_A2 = (
            EEG_reconstruida_A2 +
            bandas_A2[nombre]
        )


    # ========================================================
    # 8. ORIGINAL VS RECONSTRUIDA
    # ========================================================

    plt.figure(figsize=(14, 8))


    plt.subplot(2, 1, 1)

    plt.plot(
        t_grafica,
        A1[:muestras_grafica],
        label="Original"
    )

    plt.plot(
        t_grafica,
        EEG_reconstruida_A1[:muestras_grafica],
        "--",
        label="Reconstruida"
    )

    plt.title(
        "A1 - Original vs reconstruida mediante Fourier"
    )

    plt.xlabel("Tiempo (s)")
    plt.ylabel("Amplitud")

    plt.legend()
    plt.grid(True)


    plt.subplot(2, 1, 2)

    plt.plot(
        t_grafica,
        A2[:muestras_grafica],
        label="Original"
    )

    plt.plot(
        t_grafica,
        EEG_reconstruida_A2[:muestras_grafica],
        "--",
        label="Reconstruida"
    )

    plt.title(
        "A2 - Original vs reconstruida mediante Fourier"
    )

    plt.xlabel("Tiempo (s)")
    plt.ylabel("Amplitud")

    plt.legend()
    plt.grid(True)


    plt.tight_layout()


    plt.savefig(
        carpeta_salida /
        "07_original_vs_reconstruida.png",
        dpi=300
    )

    plt.close()


    # ========================================================
    # 9. ERROR DE RECONSTRUCCION
    # ========================================================

    error_A1 = np.mean(
        (A1 - EEG_reconstruida_A1) ** 2
    )

    error_A2 = np.mean(
        (A2 - EEG_reconstruida_A2) ** 2
    )


    # ========================================================
    # 10. GUARDAR RESULTADOS EN TXT
    # ========================================================

    archivo_txt = (
        carpeta_salida /
        "resultados.txt"
    )


    with open(
        archivo_txt,
        "w",
        encoding="utf-8"
    ) as archivo:

        archivo.write(
            "ANALISIS EEG MEDIANTE FOURIER\n"
        )

        archivo.write("=" * 60 + "\n\n")


        archivo.write(
            f"Archivo: {ruta_archivo.name}\n"
        )

        archivo.write(
            f"Muestras: {N}\n"
        )

        archivo.write(
            f"Frecuencia de muestreo: {FS} Hz\n"
        )

        archivo.write(
            f"Duracion: {duracion:.2f} segundos\n\n"
        )


        archivo.write(
            "BANDAS ANALIZADAS\n"
        )

        archivo.write("-" * 40 + "\n")


        for nombre, (
            fmin,
            fmax
        ) in BANDAS.items():

            archivo.write(
                f"{nombre}: "
                f"{fmin}-{fmax} Hz\n"
            )


        archivo.write("\n")


        archivo.write(
            "POTENCIA A1\n"
        )

        archivo.write("-" * 40 + "\n")


        for nombre in BANDAS:

            archivo.write(
                f"{nombre}: "
                f"{potencia_A1[nombre]:.8f}\n"
            )


        archivo.write("\n")


        archivo.write(
            "POTENCIA A2\n"
        )

        archivo.write("-" * 40 + "\n")


        for nombre in BANDAS:

            archivo.write(
                f"{nombre}: "
                f"{potencia_A2[nombre]:.8f}\n"
            )


        archivo.write("\n")


        archivo.write(
            "ERROR DE RECONSTRUCCION\n"
        )

        archivo.write("-" * 40 + "\n")


        archivo.write(
            f"A1: {error_A1:.8f}\n"
        )

        archivo.write(
            f"A2: {error_A2:.8f}\n"
        )


    # ========================================================
    # 11. MENSAJE EN TERMINAL
    # ========================================================

    print("\nResultados guardados en:")

    print(carpeta_salida)

    print("\nPotencia A1:")

    for nombre in BANDAS:

        print(
            f"  {nombre}: "
            f"{potencia_A1[nombre]:.6f}"
        )

    print("\nPotencia A2:")

    for nombre in BANDAS:

        print(
            f"  {nombre}: "
            f"{potencia_A2[nombre]:.6f}"
        )

    print(
        f"\nError reconstruccion A1: "
        f"{error_A1:.8f}"
    )

    print(
        f"Error reconstruccion A2: "
        f"{error_A2:.8f}"
    )


# ============================================================
# 12. BUSCAR TODOS LOS ARCHIVOS EEG
# ============================================================

print("\n")
print("=" * 70)
print("ANALISIS DE ARCHIVOS EEG")
print("=" * 70)

print(f"\nCarpeta de datos:")
print(CARPETA_DATOS)


# Buscar todos los .txt
archivos_EEG = sorted(
    CARPETA_DATOS.glob("*.txt")
)


# Verificar
if len(archivos_EEG) == 0:

    print(
        "\nNO SE ENCONTRARON ARCHIVOS .TXT"
    )

    print(
        "\nVerifica que tus archivos esten dentro de:"
    )

    print(CARPETA_DATOS)

    raise SystemExit


print(
    f"\nSe encontraron "
    f"{len(archivos_EEG)} archivos EEG:"
)


for archivo in archivos_EEG:

    print(
        f"  - {archivo.name}"
    )


# ============================================================
# 13. PROCESAR TODOS LOS ARCHIVOS
# ============================================================

for archivo in archivos_EEG:

    procesar_archivo(archivo)


# ============================================================
# 14. FINAL
# ============================================================

print("\n")
print("=" * 70)
print("TODOS LOS ARCHIVOS HAN SIDO PROCESADOS")
print("=" * 70)

print(
    "\nLos resultados estan en:"
)

print(
    CARPETA_RESULTADOS
)